In [38]:
import torch
import torch.nn.functional as F
import requests
import random
from torch.utils.data import DataLoader, TensorDataset

In [39]:
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
words = requests.get(url).text.splitlines()
print(words.__len__())
print(len(words))



random.shuffle(words)
print(words[:10])
len(words)

32033
32033
['niko', 'emyiah', 'kyndell', 'winry', 'tahiya', 'mariya', 'alfonso', 'niyah', 'edahi', 'ronit']


32033

In [40]:
if device := torch.device("mps" if torch.backends.mps.is_available() else "cpu"):
    print(f"Using device: {device}")

block_size = 7
epochs = 200
lr = 0.001

Using device: mps


In [41]:
chars = ['.'] + sorted(set(''.join(words)))
def stoi(s):
    return {c: i for i, c in enumerate(chars)}[s]
def itos(i):
    return {i: c for i, c in enumerate(chars)}[i]
print(itos(1), itos(2), itos(3), itos(0))
print(stoi('.'), stoi('a'), stoi('b'), stoi('c'))

a b c .
0 1 2 3


In [42]:
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi(ch)
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [43]:
X, Y = build_dataset(words)
n = X.shape[0]
Xtr, Ytr = X[:int(n*0.9)], Y[:int(n*0.9)]
Xdev, Ydev = X[int(n*0.9):], Y[int(n*0.9):]
Xtr, Ytr = [x.to(device) for x in (Xtr, Ytr)]
Xdev, Ydev = [x.to(device) for x in (Xdev, Ydev)]
print(X.shape, Y.shape)
print(Xtr.shape, Ytr.shape)
print(Xdev.shape, Ydev.shape)

torch.Size([228146, 7]) torch.Size([228146])
torch.Size([205331, 7]) torch.Size([205331])
torch.Size([22815, 7]) torch.Size([22815])


In [44]:
trainds = torch.utils.data.TensorDataset(Xtr.cpu(), Ytr.cpu())
traindl = DataLoader(
    dataset=trainds,
    batch_size=1024,
    pin_memory=False,
    shuffle=True
)

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        
        